## Imports

In [1]:
import json
from typing import Annotated

import requests
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import InjectedToolArg, tool
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

/home/dhruv-kapri/Desktop/Projects/langchain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## tool create

In [2]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """
    url = f"https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}"

    response = requests.get(url)

    return response.json()


@tool
def convert(
    base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]
) -> float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    """

    return base_currency_value * conversion_rate


In [3]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

In [4]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1767484801,
 'time_last_update_utc': 'Sun, 04 Jan 2026 00:00:01 +0000',
 'time_next_update_unix': 1767571201,
 'time_next_update_utc': 'Mon, 05 Jan 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 90.1096}

In [5]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

## Tool Binding

In [6]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [7]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

## Messages

In [8]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)

In [9]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': '4bc45178-057f-4169-ac66-b321e5ab07a2',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'bb039dfa-f0d7-4f55-8348-db6969fbdc17',
  'type': 'tool_call'}]

In [10]:
for tool_call in ai_message.tool_calls:
    if tool_call['name'] == 'get_conversion_factor':
        tool_message = get_conversion_factor.invoke(tool_call)
        conversion_rate = json.loads(tool_message.content)['conversion_rate']
        messages.append(tool_message)

    if tool_call['name'] == 'convert':
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message = convert.invoke(tool_call)
        messages.append(tool_message)


In [11]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'convert', 'arguments': '{"base_currency_value": 10}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b8947-1768-7bb2-9f73-0253987ae499-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': '4bc45178-057f-4169-ac66-b321e5ab07a2', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10, 'conversion_rate': 0.0111}, 'id': 'bb039dfa-f0d7-4f55-8348-db6969fbdc17', 'type': 'tool_call'}], usage_metadata={'input_tokens': 150, 'output_tokens': 44, 'total_tokens': 194, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='{"result": "success", "documentation": "h

In [12]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR and USD is 0.0111. 10 INR is equal to 0.111 USD.'